# TEMPO Virtual Datacube Generation with `Icechunk`

## Overview

Create an Icechunk store for a virtual datacube of TEMPO satellite data that is ready for performing analyses without downloading files.

## Quick Start

**To run this notebook:**
1. Update configuration parameters in the Configuration section below
2. Run all cells in order
3. Find your Icechunk store at the specified output location

## Prerequisites
- **NASA Earthdata account** ([sign up here](https://urs.earthdata.nasa.gov/users/new))
- **earthaccess**: NASA data search and authentication
- **virtualizarr**: Virtual dataset creation
- **xarray**: Multidimensional data handling
- **obstore**: Cloud storage access
- **icechunk**: Version-controlled virtual zarr store

## Virtual vs Traditional Approach

* **Traditional:** Download files (GB-TB) → Load into memory → Analyze 
* **Virtual:** Create lightweight reference files (MB) → Access on-demand (without downloading) → Analyze at scale

---

## 1. Setup and Authentication

In [1]:
import tempfile
import warnings
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse

import earthaccess
import xarray as xr
import virtualizarr as vz
from tqdm.notebook import tqdm

# VirtualiZarr components
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from virtualizarr.registry import ObjectStoreRegistry

# Object store components  
from obstore.store import S3Store
from obstore.auth.earthdata import NasaEarthdataCredentialProvider

# Icechunk and storage utilities
import icechunk as ic
from icechunk import IcechunkStore, S3StaticCredentials, s3_storage

In [2]:
import virtualizarr
import zarr
import obstore
import fsspec

for l in (virtualizarr, zarr, xr, fsspec, obstore):
    print(l, l.__version__)

<module 'virtualizarr' from '/srv/conda/envs/notebook/lib/python3.11/site-packages/virtualizarr/__init__.py'> 2.1.2
<module 'zarr' from '/srv/conda/envs/notebook/lib/python3.11/site-packages/zarr/__init__.py'> 3.1.3
<module 'xarray' from '/srv/conda/envs/notebook/lib/python3.11/site-packages/xarray/__init__.py'> 2025.9.0
<module 'fsspec' from '/srv/conda/envs/notebook/lib/python3.11/site-packages/fsspec/__init__.py'> 2025.9.0
<module 'obstore' from '/srv/conda/envs/notebook/lib/python3.11/site-packages/obstore/__init__.py'> 0.8.2


### Configuration

In [3]:
# Configuration
# Modify these settings as needed

# Dataset parameters
DATASET_SHORT_NAME = "TEMPO_NO2_L3"
DATASET_VERSION = "V04"
TEMPORAL_RANGE = ("2026-01-03 12:00", "2026-01-05 12:00")  # Adjust date range as needed

# Processing parameters - to create the datacube
GROUP_NAMES = ["", "product", "geolocation", "support_data", "qa_statistics"]
CONCATENATION_DIMENSION = "time"
XARRAY_CONCAT_OPTIONS = {
    "coords": "minimal",
    "compat": "override",
    "combine_attrs": "override",
}

# Output configuration
USE_LOCAL_STORAGE = True  # Set False for S3 storage
LOCAL_TEMP_DIR = "tempo_icechunk"  # None = auto-generate temp directory
OUTPUT_PREFIX = "tempo_icechunk"

# Constants (typically don't need modification)
S3_REGION = "us-west-2"
CREDENTIAL_ENDPOINTS = {
    "asdc-prod-protected": "https://data.asdc.earthdata.nasa.gov/s3credentials",
}

#### Earthdata Login and data provider credentials

In [4]:
try:
    auth = earthaccess.login()
    print(f"✓ Authenticated as {auth.username}" if auth.authenticated else "❌ Authentication failed")
except Exception as e:
    print(f"❌ Login failed: {e}")
    print("Check credentials, or visit: https://urs.earthdata.nasa.gov/users/new for account setup")
    raise

✓ Authenticated as dkaufas


In [5]:
# We can use the NASA credential provider to automatically refresh credentials
try:
    credential_provider = NasaEarthdataCredentialProvider(CREDENTIAL_ENDPOINTS["asdc-prod-protected"])
    print("✓ Credential provider initialized")
except Exception as e:
    print(f"❌ Failed to initialize credential provider: {e}")
    raise

✓ Credential provider initialized


## 2. Build Virtual Datacube

Create a virtual representation of TEMPO data without downloading files:

1. **Search** cloud-hosted NetCDF files
2. **Create** virtual datasets for each file and group
3. **Combine** into unified structure

**TEMPO Data Structure**

- *Root group* (`""`): Global metadata and coordinates
- *Geolocation group*: Spatial reference data
- *Product group*: Primary science variables
- *Support Data group*: Ancillary information
- *QA Statistics group*: Quality metrics

We'll process groups separately, then merge them.

### 2.1. Search NASA Data Archive

Search for TEMPO satellite files in NASA's cloud storage.

In [6]:
# Retrieve data files for the dataset I'm interested in
print(f"Searching for TEMPO Level 3 data...")
search_params = {
    "short_name": DATASET_SHORT_NAME,
    "version": DATASET_VERSION,
    "cloud_hosted": True,
}

# Add temporal filter if configured
if 'TEMPORAL_RANGE' in globals():
    search_params["temporal"] = TEMPORAL_RANGE

results = earthaccess.search_data(**search_params)

if not results:
    raise ValueError(
        f"No data found for {DATASET_SHORT_NAME} {DATASET_VERSION}. "
        f"Verify the dataset name, version, and temporal range."
    )
print(f"✓ Found {len(results)} granules")

Searching for TEMPO Level 3 data...
✓ Found 22 granules


In [7]:
# Get S3 endpoints for all files:
s3_data_links = [g.data_links(access="direct")[0] for g in results]
parsed_s3_url = urlparse(s3_data_links[0])

print("First few granules: \n  {}".format('\n  '.join(s3_data_links[0:3])))
print(f"Parsed URL: {parsed_s3_url}")
print(f"Bucket: {parsed_s3_url.netloc}")

s3_store = S3Store(
    bucket=parsed_s3_url.netloc,
    region="us-west-2",
    credential_provider=credential_provider,
    virtual_hosted_style_request=False,
    client_options={"allow_http": True},
)
object_registry = ObjectStoreRegistry({f"s3://{parsed_s3_url.netloc}": s3_store})

s3_store

First few granules: 
  s3://asdc-prod-protected/TEMPO/TEMPO_NO2_L3_V04/2026.01.03/TEMPO_NO2_L3_V04_20260103T132255Z_S002.nc
  s3://asdc-prod-protected/TEMPO/TEMPO_NO2_L3_V04/2026.01.03/TEMPO_NO2_L3_V04_20260103T140303Z_S003.nc
  s3://asdc-prod-protected/TEMPO/TEMPO_NO2_L3_V04/2026.01.03/TEMPO_NO2_L3_V04_20260103T144311Z_S004.nc
Parsed URL: ParseResult(scheme='s3', netloc='asdc-prod-protected', path='/TEMPO/TEMPO_NO2_L3_V04/2026.01.03/TEMPO_NO2_L3_V04_20260103T132255Z_S002.nc', params='', query='', fragment='')
Bucket: asdc-prod-protected


S3Store(bucket="asdc-prod-protected")

In [8]:
# Optional: Download original files for comparison
# files_downloaded = earthaccess.download(results[0])
# dtree = xr.open_datatree(files_downloaded[0])
# dtree

### 2.2. Set up virtual dataset creation function

In [9]:
def create_virtual_dataset(file_path: str, group_name: str):
    """Create virtual dataset from NetCDF group without loading data.

    Without downloading files, build lightweight references to cloud data, containing:
        - **Metadata**: Dimensions, coordinates, attributes
        - **Chunk references**: Pointers to specific byte ranges in cloud storage
        - **No actual data**: Just instructions on how to fetch it when needed

    Parameters
    ----------
    file_path : str
        S3 URL to the NetCDF file.
    group_name : str
        HDF5 group name to read (e.g., "product", "geolocation").

    Returns
    -------
    xr.Dataset
        Virtual dataset containing metadata and chunk references.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Numcodecs codecs*", category=UserWarning)
        return open_virtual_dataset(
            file_path, 
            parser=HDFParser(group=group_name),
            registry=object_registry
        )

### 2.3. Test a single file

In [10]:
print("Testing virtual dataset creation...")
test_vds = create_virtual_dataset(file_path=s3_data_links[0], group_name="product")
print(f"✓ Test successful. Dataset shape: {test_vds.sizes}")

Testing virtual dataset creation...
✓ Test successful. Dataset shape: Frozen({'time': 1, 'latitude': 2950, 'longitude': 7750})


### 2.4. Process All Files

In [11]:
%%time
print(f"Processing {len(s3_data_links)} files across {len(GROUP_NAMES)} groups...")

vdatasets_by_group = {group: [] for group in GROUP_NAMES}

for file_index, file_path in enumerate(tqdm(s3_data_links, desc="Processing files")):
    for group_name in tqdm(GROUP_NAMES, desc=f"Groups (file {file_index+1})", leave=False):
        virtual_ds = create_virtual_dataset(file_path, group_name)
        vdatasets_by_group[group_name].append(virtual_ds)
        
print(f"✓ Created {sum(len(d) for d in vdatasets_by_group.values())} virtual datasets")

Processing 22 files across 5 groups...


Processing files:   0%|          | 0/22 [00:00<?, ?it/s]

Groups (file 1):   0%|          | 0/5 [00:00<?, ?it/s]

Groups (file 2):   0%|          | 0/5 [00:00<?, ?it/s]

Groups (file 3):   0%|          | 0/5 [00:00<?, ?it/s]

...

Groups (file 19):   0%|          | 0/5 [00:00<?, ?it/s]

Groups (file 20):   0%|          | 0/5 [00:00<?, ?it/s]

Groups (file 21):   0%|          | 0/5 [00:00<?, ?it/s]

Groups (file 22):   0%|          | 0/5 [00:00<?, ?it/s]

✓ Created 110 virtual datasets
CPU times: user 27.5 s, sys: 30.8 s, total: 58.3 s
Wall time: 6min 10s


In [12]:
# Show what a virtual dataset looks like
print(f"\nSample virtual dataset structure:")
sample_vds = vdatasets_by_group["product"][0]
sample_vds


Sample virtual dataset structure:


<xarray.Dataset> Size: 594MB
Dimensions:                                  (time: 1, latitude: 2950,
                                              longitude: 7750)
Dimensions without coordinates: time, latitude, longitude
Data variables:
    vertical_column_troposphere              (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, chunks=(1, 7...
    vertical_column_troposphere_uncertainty  (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, chunks=(1, 7...
    vertical_column_stratosphere             (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, chunks=(1, 7...
    main_data_quality_flag                   (time, latitude, longitude) int16 46MB ManifestArray<shape=(1, 2950, 7750), dtype=int16, chunks=(1, 984,...

In [13]:
%%time
# Concatenate datasets for each group.
concatenated_by_group = {}
for group_name, group_datasets in vdatasets_by_group.items():
    concatenated_by_group[group_name] = xr.concat(
        group_datasets, dim=CONCATENATION_DIMENSION, **XARRAY_CONCAT_OPTIONS
    )
    print(f"  ✓ Group '{group_name}': {concatenated_by_group[group_name].sizes}")

  ✓ Group '': Frozen({'time': 22, 'latitude': 2950, 'longitude': 7750})
  ✓ Group 'product': Frozen({'time': 22, 'latitude': 2950, 'longitude': 7750})
  ✓ Group 'geolocation': Frozen({'time': 22, 'latitude': 2950, 'longitude': 7750})
  ✓ Group 'support_data': Frozen({'time': 22, 'latitude': 2950, 'longitude': 7750})
  ✓ Group 'qa_statistics': Frozen({'time': 22, 'latitude': 2950, 'longitude': 7750})
CPU times: user 87.5 ms, sys: 0 ns, total: 87.5 ms
Wall time: 87.1 ms


<timed exec>:4: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
/srv/conda/envs/notebook/lib/python3.11/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


In [14]:
# Optional: Free memory from intermediate datasets
# del vdatasets_by_group

In [15]:
concatenated_by_group["product"]

<xarray.Dataset> Size: 13GB
Dimensions:                                  (time: 22, latitude: 2950,
                                              longitude: 7750)
Dimensions without coordinates: time, latitude, longitude
Data variables:
    vertical_column_troposphere              (time, latitude, longitude) float64 4GB ManifestArray<shape=(22, 2950, 7750), dtype=float64, chunks=(1, 738...
    vertical_column_troposphere_uncertainty  (time, latitude, longitude) float64 4GB ManifestArray<shape=(22, 2950, 7750), dtype=float64, chunks=(1, 738...
    vertical_column_stratosphere             (time, latitude, longitude) float64 4GB ManifestArray<shape=(22, 2950, 7750), dtype=float64, chunks=(1, 738...
    main_data_quality_flag                   (time, latitude, longitude) int16 1GB ManifestArray<shape=(22, 2950, 7750), dtype=int16, chunks=(1, 984, ...

In [16]:
%%time
# Merge the groups into one dataset.
datacube = xr.merge([vds for _, vds in concatenated_by_group.items()])

print(f"✓ Consolidated datacube created with {len(datacube.data_vars)} variables")
print(f"  Time dimension: {datacube.sizes.get('time', 'N/A')} steps")

datacube

✓ Consolidated datacube created with 36 variables
  Time dimension: 22 steps
CPU times: user 1.26 ms, sys: 0 ns, total: 1.26 ms
Wall time: 1.26 ms


<xarray.Dataset> Size: 100GB
Dimensions:                                              (time: 22,
                                                          latitude: 2950,
                                                          longitude: 7750)
Coordinates:
  * longitude                                            (longitude) float32 31kB ...
  * latitude                                             (latitude) float32 12kB ...
  * time                                                 (time) datetime64[ns] 176B ...
Data variables: (12/36)
    weight                                               (time, latitude, longitude) float32 2GB ManifestArray<shape=(22, 2950, 7750), dtype=float32, ch...
    vertical_column_troposphere                          (time, latitude, longitude) float64 4GB ManifestArray<shape=(22, 2950, 7750), dtype=float64, ch...
    vertical_column_troposphere_uncertainty              (time, latitude, longitude) float64 4GB ManifestArray<shape=(22, 2950, 7750), dtype=float64, ch...
    vertical_column_stratosphere                         (time, latitude, longitude) float64 4GB ManifestArray<shape=(22, 2950, 7750), dtype=float64, ch...
    main_data_quality_flag                               (time, latitude, longitude) int16 1GB ManifestArray<shape=(22, 2950, 7750), dtype=int16, chun...
    solar_zenith_angle                                   (time, latitude, longitude) float32 2GB ManifestArray<shape=(22, 2950, 7750), dtype=float32, ch...
    ...                                                   ...
    num_vertical_column_stratosphere_samples             (time, latitude, longitude) int32 2GB ManifestArray<shape=(22, 2950, 7750), dtype=int32, chun...
    min_vertical_column_stratosphere_sample              (time, latitude, longitude) float64 4GB ManifestArray<shape=(22, 2950, 7750), dtype=float64, ch...
    max_vertical_column_stratosphere_sample              (time, latitude, longitude) float64 4GB ManifestArray<shape=(22, 2950, 7750), dtype=float64, ch...
    num_vertical_column_total_samples                    (time, latitude, longitude) int32 2GB ManifestArray<shape=(22, 2950, 7750), dtype=int32, chun...
    min_vertical_column_total_sample                     (time, latitude, longitude) float64 4GB ManifestArray<shape=(22, 2950, 7750), dtype=float64, ch...
    max_vertical_column_total_sample                     (time, latitude, longitude) float64 4GB ManifestArray<shape=(22, 2950, 7750), dtype=float64, ch...
Attributes: (12/40)
    history:                          2026-01-03T16:59:48Z: L2_regrid -v /pro...
    scan_num:                         2
    time_coverage_start:              2026-01-03T13:22:55Z
    time_coverage_end:                2026-01-03T14:02:44Z
    time_coverage_start_since_epoch:  1451481793.0205176
    time_coverage_end_since_epoch:    1451484182.6488934
    ...                               ...
    title:                            TEMPO Level 3 nitrogen dioxide product
    collection_shortname:             TEMPO_NO2_L3
    collection_version:               1
    keywords:                         EARTH SCIENCE>ATMOSPHERE>AIR QUALITY>NI...
    summary:                          Nitrogen dioxide Level 3 files provide ...
    coremetadata:                     \nGROUP                  = INVENTORYMET...

## 3. Export to Icechunk

Following: https://icechunk.io/en/latest/quickstart/#installation

### 3.1. Helper functions

Adapted from [icechunk_opener.py](https://github.com/jbusecke/earthaccess/blob/icechunk_opener/earthaccess/icechunk_opener.py) by Julius Busecke.

These functions handle authentication with NASA Earthdata for accessing
virtual chunk data through Icechunk repositories.

In [17]:
def _get_credential_endpoint(url: str) -> str:
    sep = "/"
    parsed = urlparse(url)
    if parsed.scheme != "s3":
        raise ValueError(
            "Only s3 is supported as storage protocol. Got {parsed.protocol}"
        )
        # Note: Consider whether "protocol" is the correct terminology here.
    bucket_w_prefix_full = parsed.netloc + parsed.path.rstrip(sep)
    components = bucket_w_prefix_full.split(sep)

    while len(components) > 0:
        partial_target = sep.join(components)
        if partial_target in CREDENTIAL_ENDPOINTS.keys():
            return CREDENTIAL_ENDPOINTS[partial_target]
        components = components[0:-1]

    raise ValueError("Could not find any matching credential endpoint for {url}")

In [18]:
class S3IcechunkCredentials:
    def __init__(self, endpoint: str):
        self.endpoint = endpoint

    def __call__(self) -> S3StaticCredentials:
        if not earthaccess.__auth__.authenticated:
            raise ValueError(
                "A valid Earthdata login instance is required to retrieve credentials for icechunk stores"
            )
        creds = earthaccess.__auth__.get_s3_credentials(endpoint=self.endpoint)
        if len(creds) == 0:
            raise ValueError(f"Got no credentials from endpoint {self.endpoint}")
        return S3StaticCredentials(
            access_key_id=creds["accessKeyId"],
            secret_access_key=creds["secretAccessKey"],
            expires_after=datetime.fromisoformat(creds["expiration"]),
            session_token=creds["sessionToken"],
        )

In [19]:
def get_virtual_chunk_credentials(
    storage: ic.Storage,
) -> dict[str, ic.AnyCredential | None]:
    """Function to retrieve virtual chunk containers from icechunk storage and authenticate
    all allowed virtual chunk prefixes using EDL credentials.
    """
    # get config and extract virtual containers
    config = ic.Repository.fetch_config(storage=storage)
    # TODO: accommodate case without virtual chunk containers.
    if not config:
        msg = f"Got empty config from {storage=} and will not try to infer chunk containers. If this is a native store this behavior is expected if the config was not persisted to storage."
        warnings.warn(message=msg)
        vchunk_container_urls: List[str] = []
    else:
        vchunk_containers = config.virtual_chunk_containers
        if vchunk_containers:
            vchunk_container_urls = list(vchunk_containers.keys())
        else:
            raise ValueError("No virtual chunk containers found.")

    # try to build authentication for all virtual chunk containers. If any of the virtual
    # chunk containers is not 'approved' it will raise an error in `_get_credential_endpoint`.
    # We will catch the error here, warn, and only return the authenticated urls.
    # Users will then get an error for the remaining containers and need to add those manually!
    failed_container_urls = []
    credential_mapping = {}
    for url in vchunk_container_urls:
        try:
            endpoint = _get_credential_endpoint(url)
            credential_mapping[url] = ic.s3_refreshable_credentials(
                S3IcechunkCredentials(endpoint=endpoint)
            )
        except ValueError:
            failed_container_urls.append(url)

    if len(failed_container_urls) > 0:
        # TODO: link to credentials in icechunk + docs about the endpoint registry
        warnings.warn(
            f"Could not build virtual chunk credentials for {failed_container_urls}.\
                      If the URL is a non EDL bucket, you have to manually construct credentials (...)"
        )

    # TODO: Check how easy it is to 'splice' this output with manually created credentials
    return ic.containers_credentials(credential_mapping)

### 3.2. Configure Storage Location

In [20]:
# Parse source data URL for virtual chunk container
parsed = urlparse(s3_data_links[0])
bucket = parsed.netloc
prefix = parsed.path.lstrip("/")
bucket_path = f"s3://{bucket}/"

print(f"Virtual chunk container (bucket): {bucket_path}")

# Get credentials for accessing virtual chunks
endpoint = _get_credential_endpoint(s3_data_links[0])
cred_provider = S3IcechunkCredentials(endpoint=endpoint)
static_creds = cred_provider()

virtual_credentials = ic.containers_credentials({
    bucket_path: ic.s3_credentials(
        access_key_id=static_creds.access_key_id, 
        secret_access_key=static_creds.secret_access_key
    )
})

print(f"✓ Credentials configured for {bucket_path}")

Virtual chunk container (bucket): s3://asdc-prod-protected/
✓ Credentials configured for s3://asdc-prod-protected/


In [21]:
# Initialize Icechunk configuration
config = ic.RepositoryConfig.default()

config.set_virtual_chunk_container(
    ic.VirtualChunkContainer(
        bucket_path,
        store=ic.s3_store(region="us-west-2"),
    )
)

In [22]:
# Define the S3 Virtual Chunk Container
storage = s3_storage(
    bucket=bucket,
    prefix=prefix,
    region="us-west-2",
    get_credentials=cred_provider,
)

### 3.3. Create Repository Storage

In [23]:
if USE_LOCAL_STORAGE:
    # Local storage (for testing/development)
    if LOCAL_TEMP_DIR:
        storage_path = Path(LOCAL_TEMP_DIR)
        storage_path.mkdir(parents=True, exist_ok=True)
    else:
        tempdir = tempfile.TemporaryDirectory()
        storage_path = Path(tempdir.name)
    
    repo_storage = ic.local_filesystem_storage(str(storage_path))
    print(f"✓ Using local storage: {storage_path}")
else:
    # S3 storage (for production/sharing)
    # TODO: Add S3 storage configuration
    raise NotImplementedError("S3 storage not yet configured")

✓ Using local storage: tempo_icechunk  2026-02-10T22:13:12.683094Z  WARN icechunk::storage::object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk/src/storage/object_store.rs:81




Note: Temporary directory will be cleaned up when Python exits

For persistent storage, set `LOCAL_TEMP_DIR` to a permanent location

In [24]:
# Create/Open repository
repo = ic.Repository.open_or_create(
    storage=repo_storage,
    config=config,
    authorize_virtual_chunk_access=virtual_credentials,
)

### 3.4. Write (commit) the virtual dataset into icechunk

#### Write to **new** repo

In [25]:
# try:
#     # Create a writable session on the default main branch.
#     session = repo.writable_session("main")
#     # Send the virtual references to the Icechunk store, and commit them.
#     datacube.vz.to_icechunk(session.store)

#     commit_msg = f"Initial datacube creation - {DATASET_SHORT_NAME} - {datetime.now().strftime('%Y-%m-%d')}"
#     commit_id = session.commit(commit_msg)
#     print(f"✓ Datacube committed successfully")
#     print(f"  Commit ID: {commit_id}")
    
# except Exception as e:
#     print(f"❌ Error writing to Icechunk: {e}")
#     raise

#### Write to **existing** repo


In [26]:
datacube["num_vertical_column_stratosphere_samples"]

<xarray.DataArray 'num_vertical_column_stratosphere_samples' (time: 22,
                                                              latitude: 2950,
                                                              longitude: 7750)> Size: 2GB
ManifestArray<shape=(22, 2950, 7750), dtype=int32, chunks=(1, 984, 2584)>
Coordinates:
  * longitude  (longitude) float32 31kB -168.0 -168.0 -167.9 ... -13.03 -13.01
  * latitude   (latitude) float32 12kB 14.01 14.03 14.05 ... 72.95 72.97 72.99
  * time       (time) datetime64[ns] 176B 2026-01-03T13:23:13.020517632 ... 2...
Attributes:
    comment:     Number of Level 2 pixel values contributing to the area-weig...
    _FillValue:  -1

In [27]:
# Remove the 'units' attribute
del datacube["num_vertical_column_stratosphere_samples"].attrs["_FillValue"]

In [28]:
datacube["num_vertical_column_stratosphere_samples"]

<xarray.DataArray 'num_vertical_column_stratosphere_samples' (time: 22,
                                                              latitude: 2950,
                                                              longitude: 7750)> Size: 2GB
ManifestArray<shape=(22, 2950, 7750), dtype=int32, chunks=(1, 984, 2584)>
Coordinates:
  * longitude  (longitude) float32 31kB -168.0 -168.0 -167.9 ... -13.03 -13.01
  * latitude   (latitude) float32 12kB 14.01 14.03 14.05 ... 72.95 72.97 72.99
  * time       (time) datetime64[ns] 176B 2026-01-03T13:23:13.020517632 ... 2...
Attributes:
    comment:  Number of Level 2 pixel values contributing to the area-weighte...

In [29]:
try:
    # Create a writable session on the default main branch.
    commit_msg = f"Wrote some data - {DATASET_SHORT_NAME} - {datetime.now().strftime('%Y-%m-%d')}"

    # Use the .transaction function as a context manager, which automatically commits when the context exits.
    with repo.transaction(branch="main", message=commit_msg) as store:

        # Append virtual references to existing array (along 'time' dimension)
        # 'datacube' is the new dataset to append
        # datacube.to_zarr(store, mode="a", append_dim="time", consolidated=False)
        
        # NEW CODE
        datacube.vz.to_icechunk(store, append_dim="time")
        
        print(f"✓ Datacube committed successfully")
        print(f"  Commit ID: {commit_id}")
    
except Exception as e:
    print(f"❌ Error writing to Icechunk: {e}")
    raise

/srv/conda/envs/notebook/lib/python3.11/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


✓ Datacube committed successfully
❌ Error writing to Icechunk: name 'commit_id' is not defined


NameError: name 'commit_id' is not defined

In [ ]:
# Verify the commit
session = repo.readonly_session("main")
verification_ds = xr.open_zarr(session.store)
assert len(verification_ds.data_vars) == len(datacube.data_vars), "Variable count mismatch"
print(f"  Variables: {len(verification_ds.data_vars)}")
print(f"  Time steps: {verification_ds.sizes.get('time', 'N/A')}")

## 4. Load and Visualize Datacube

This section demonstrates how to:
1. Re-open an existing Icechunk repository
2. Access the virtual datacube
3. Create visualizations using cloud data on-demand

In [ ]:
import zarr

In [ ]:
# Open authenticated icechunk repo
virtual_chunk_credentials = get_virtual_chunk_credentials(repo_storage)

repo = ic.Repository.open(
    storage=repo_storage, 
    authorize_virtual_chunk_access=virtual_chunk_credentials
)

In [ ]:
# View all snapshots/commits on the main branch
for ancestor in repo.ancestry(branch="main"):
    print(ancestor)

In [ ]:
# We can also list all branches in the repository.
print(repo.list_branches())

In [ ]:
# Open a read-only session on the 'main' branch
session = repo.readonly_session("main")

# Access the Zarr store from this session
store = session.store

In [ ]:
# Open the root group
root = zarr.open_group(store=store, mode="r")

# List all arrays and groups in the store
print(list(root.array_keys()))   # Shows array names
print(list(root.group_keys()))   # Shows sub-group names
print(root.tree())               # Shows a visual hierarchy (if using zarr-python)

In [ ]:
ds = xr.open_zarr(store)
ds

In [ ]:
# Get min and max of time
min_time = ds['time'].min()
max_time = ds['time'].max()

# Format to seconds precision (%Y-%m-%d %H:%M:%S)
# .dt accessor works on scalar Timestamps/datetime64
formatted_min = min_time.dt.strftime("%Y-%m-%d %H:%M:%S").item()
formatted_max = max_time.dt.strftime("%Y-%m-%d %H:%M:%S").item()

print(f"Datetime range:\n\t{formatted_min}\n\tto\n\t{formatted_max}")

In [ ]:
data_variable = ds["vertical_column_troposphere"]
data_variable

### 4.2. Create visualization

This visualization demonstrates that virtual datacube access works similarly to standard xarray usage.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from xarray.plot.utils import label_from_attrs

In [ ]:
projection = ccrs.PlateCarree()
# projection = ccrs.Orthographic(0, 90)

def make_nice_map(axis):
    axis.add_feature(cfeature.OCEAN, color='lightblue')
    axis.add_feature(cfeature.COASTLINE, edgecolor='grey')
    axis.add_feature(cfeature.BORDERS, edgecolor="grey", linewidth=0.5)

    grid = axis.gridlines(draw_labels=["left", "bottom"], dms=True, linestyle=':')
    grid.xformatter = LONGITUDE_FORMATTER
    grid.yformatter = LATITUDE_FORMATTER

In [ ]:
print("Creating sample visualization from virtual datacube...")
print("Note: Data is read on-demand from cloud storage during plotting")

# Extract data for plotting
time_idx = 3
time_value = ds["time"].values[time_idx]
print(f"Plotting time step {time_idx}: {time_value}")

x = ds["longitude"].squeeze()
y = ds["latitude"].squeeze()
values = data_variable[time_idx, :, :].squeeze()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={"projection": projection})

make_nice_map(ax)

contour_handle = ax.contourf(
    x, y, values,
    transform=ccrs.PlateCarree(),
    alpha=0.8,
    zorder=2,
)
plt.title("TEMPO NO₂ L3 V04\\nData accessed on-demand from cloud storage")

cb = plt.colorbar(contour_handle)
cb.set_label(label_from_attrs(values))

# plt.show()
plt.savefig("icechunk_demo_vis_tempo.png", dpi=250)

## 5. Cleanup

If using temporary local storage, the Icechunk store will be deleted when Python exits.

To preserve your store:
- Set `LOCAL_TEMP_DIR` to a permanent path before running, or
- Copy the contents of the temporary directory before closing this session

To delete a persistent local store:
```python
import shutil
shutil.rmtree(storage_path)